In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize

# -------------------------------------------------------------
# 1. SYNTHETIC GRAVITATIONAL LENS DATA GENERATOR
# (Simulates: 0: No Substructure, 1: Subhalo, 2: Vortex)
# -------------------------------------------------------------
def generate_synthetic_lenses(num_samples=600):
    np.random.seed(42)
    images = np.zeros((num_samples, 1, 64, 64), dtype=np.float32)
    labels = np.zeros(num_samples, dtype=np.int64)

    y, x = np.ogrid[-32:32, -32:32]
    radius = np.sqrt(x**2 + y**2)

    for i in range(num_samples):
        label = i % 3
        labels[i] = label

        # Base Einstein Ring (circular light ring)
        ring = np.exp(-((radius - 18) ** 2) / (2 * 3.5**2))

        if label == 1:
            # Subhalo: Add a localized bright clump/distortion
            cx, cy = np.random.randint(-15, 15), np.random.randint(-15, 15)
            clump = 0.8 * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * 2.5**2))
            ring += clump
        elif label == 2:
            # Vortex / Wave DM: Add subtle wavy ripples across the ring
            ripple = 0.3 * np.sin(x / 2.0) * np.cos(y / 2.0)
            ring += ripple

        # Add Gaussian background noise (telescope sensor noise)
        noise = np.random.normal(0, 0.08, (64, 64))
        images[i, 0] = ring + noise

    return images, labels

# -------------------------------------------------------------
# 2. CUSTOM PYTORCH DATASET & DATALOADER
# -------------------------------------------------------------
class LensDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Create dataset and 80/20 Train/Validation split
raw_images, raw_labels = generate_synthetic_lenses(num_samples=600)
split = int(0.8 * len(raw_images))

train_loader = DataLoader(LensDataset(raw_images[:split], raw_labels[:split]), batch_size=32, shuffle=True)
val_loader = DataLoader(LensDataset(raw_images[split:], raw_labels[split:]), batch_size=32, shuffle=False)

# -------------------------------------------------------------
# 3. ADAPTING RESNET-18 FOR DEEPLENSE
# -------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet18(weights=None)
# Adapt input layer: Grayscale (1-channel), not RGB (3-channels)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
# Adapt output layer: 3 classes (No-sub, Subhalo, Vortex)
model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------------------------------------
# 4. TRAINING & VALIDATION LOOP (5 Epochs)
# -------------------------------------------------------------
epochs = 5
train_losses, val_losses = [], []

print(f"Training on device: {device}...")
for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * images.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)

    val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# -------------------------------------------------------------
# 5. SCIENTIFIC EVALUATION: ROC-AUC CURVES & CONFUSION MATRIX
# -------------------------------------------------------------
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1) # Convert logits to probabilities
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(torch.argmax(probs, dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

# Plot 1: Loss Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(range(1, epochs + 1), train_losses, label='Train Loss')
plt.plot(range(1, epochs + 1), val_losses, label='Val Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot 2: Confusion Matrix
plt.subplot(1, 3, 2)
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No-Sub', 'Subhalo', 'Vortex'])
disp.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title('Confusion Matrix')

# Plot 3: Multi-Class One-vs-Rest ROC-AUC Curves
plt.subplot(1, 3, 3)
labels_bin = label_binarize(all_labels, classes=[0, 1, 2])
class_names = ['No-Sub', 'Subhalo', 'Vortex']

for i in range(3):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('Multi-Class ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')

plt.tight_layout()
plt.savefig('deeplense_evaluation_plots.png')
plt.show()
print("Evaluation plots saved as 'deeplense_evaluation_plots.png'!")